# 12C — Blinded Prompt Calibration

This notebook compares the original conservative extraction contract with the analyst-style v2 contract on the **same 25 SEC documents**. Selection is frozen without returns, embeddings, company names, or API outputs. The comparison measures evidence fidelity and classification discipline—not which prompt produces more positive labels. No portfolio or return outcome is loaded here.

## 1. Imports, paths, and immutable configuration

The API is disabled by default. Responses are checkpointed, so interruption is recoverable. Pricing is deliberately entered by the researcher because model prices can change.

In [6]:
from pathlib import Path
import hashlib, json, os, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name != 'sp500_critical_minerals':
    candidate = ROOT / 'geopolitical_exposure' / 'sp500_critical_minerals'
    if candidate.exists(): ROOT = candidate
PARENT = ROOT.parent
LEGACY_CONFIG = PARENT / 'quarterly_sec_pilot' / 'config'
CONFIG = ROOT / 'config'
DATA = ROOT / 'data' / 'processed'
CAL = ROOT / 'prompt_calibration'
RESPONSES = CAL / 'responses'
MANIFESTS = CAL / 'manifests'
for path in [CAL, RESPONSES, MANIFESTS]: path.mkdir(parents=True, exist_ok=True)
CALIBRATION_N = 25
CALIBRATION_SEED = 20240903
OPENAI_MODEL = os.getenv('OPENAI_SP500_CALIBRATION_MODEL', 'gpt-5.6-terra')
RUN_CALIBRATION = True
INPUT_USD_PER_MILLION = np.nan
OUTPUT_USD_PER_MILLION = np.nan
print({'root': str(ROOT), 'model': OPENAI_MODEL, 'run_enabled': RUN_CALIBRATION})

{'root': '/Users/harmandeep/Desktop/Projects/Asset_Embeddings/geopolitical_exposure/sp500_critical_minerals', 'model': 'gpt-5.6-terra', 'run_enabled': True}


## 2. Load and validate both frozen prompt contracts

Every object in a strict schema must require all declared fields. Hashes make the exact prompt and schema used in each call auditable.

In [7]:
contracts = {
    'v1': (LEGACY_CONFIG/'critical_minerals_extraction_prompt_v1.md', LEGACY_CONFIG/'critical_minerals_exposure_schema_v1.json'),
    'v2': (CONFIG/'critical_minerals_extraction_prompt_v2.md', CONFIG/'critical_minerals_exposure_schema_v2.json'),
}
def sha256_bytes(value): return hashlib.sha256(value).hexdigest()
def validate_strict_schema(node, location='root'):
    if isinstance(node, dict):
        if node.get('type') == 'object':
            props = set(node.get('properties', {})); required = set(node.get('required', []))
            assert props == required, f'{location}: required differs from properties: {props ^ required}'
            assert node.get('additionalProperties') is False, f'{location}: additionalProperties must be false'
        for key, value in node.items(): validate_strict_schema(value, f'{location}.{key}')
    elif isinstance(node, list):
        for index, value in enumerate(node): validate_strict_schema(value, f'{location}[{index}]')
loaded = {}
rows = []
for version, (prompt_path, schema_path) in contracts.items():
    prompt = prompt_path.read_text(); schema = json.loads(schema_path.read_text()); validate_strict_schema(schema)
    loaded[version] = {'prompt': prompt, 'schema': schema}
    rows.append({'version':version,'prompt_sha256':sha256_bytes(prompt.encode()),'schema_sha256':sha256_bytes(schema_path.read_bytes()),'prompt_words':len(prompt.split())})
contract_audit = pd.DataFrame(rows); display(contract_audit)
contract_audit.to_parquet(MANIFESTS/'contract_audit.parquet', index=False)

,version,prompt_sha256,schema_sha256,prompt_words
0,v1,988ec1349b5937b04b823b87a72c35fca851bea059c392...,e02da8f8be18a16180eb900bc00f4dd8096079cb728c4c...,485
1,v2,bbbdb995d52becd004fe4bcd6e2b91acf9c6d3f36bc931...,9094b93597eaf7e19140ee8f9e1a5e65bc79bc4f006c2b...,643


## 3. Freeze one common 25-document calibration sample

The sample is deterministic and balances 10-K/10-Q documents and retrieval-depth strata. It is selected before either prompt is called. Existing selection files are never silently regenerated.

In [8]:
records_path = DATA/'sp500_retrieval_records.jsonl'
queue_path = DATA/'sp500_api_queue.parquet'
assert records_path.exists(), 'Finish 12B retrieval construction first.'
records = [json.loads(line) for line in records_path.read_text().splitlines() if line.strip()]
lookup = {str(r['document_id']): r for r in records}
if queue_path.exists():
    candidates = pd.read_parquet(queue_path).copy()
else:
    candidates = pd.DataFrame([{'document_id':r['document_id'],'firm_id':r['firm_id'],'document_type':r['document_type'],'available_date':r['available_date'],'passage_count':len(r['passages']),'approx_input_tokens':r.get('retrieved_characters',0)//4} for r in records])
candidates['document_id'] = candidates.document_id.astype(str)
candidates = candidates[candidates.document_id.isin(lookup)].copy()
assert len(candidates) >= CALIBRATION_N, f'Only {len(candidates)} eligible documents exist.'
candidates['passage_count'] = candidates.document_id.map(lambda x: len(lookup[x]['passages']))
candidates['retrieval_stratum'] = pd.cut(candidates.passage_count, bins=[-1,2,6,np.inf], labels=['low','medium','high'])
selection_path = MANIFESTS/'calibration_sample_25.parquet'
if selection_path.exists():
    sample = pd.read_parquet(selection_path)
else:
    candidates['stable_order'] = candidates.document_id.map(lambda x: sha256_bytes(f'{CALIBRATION_SEED}|{x}'.encode()))
    candidates = candidates.sort_values(['document_type','retrieval_stratum','stable_order'])
    strata = [g for _,g in candidates.groupby(['document_type','retrieval_stratum'], observed=True, sort=True)]
    chosen=[]; depth=0
    while len(chosen)<CALIBRATION_N:
        added=False
        for group in strata:
            if depth < len(group) and len(chosen)<CALIBRATION_N: chosen.append(group.iloc[depth]); added=True
        if not added: break
        depth += 1
    sample = pd.DataFrame(chosen).drop(columns=['stable_order']).reset_index(drop=True)
    sample.to_parquet(selection_path,index=False)
assert sample.document_id.nunique()==CALIBRATION_N
display(sample.groupby(['document_type','retrieval_stratum'], observed=True).size().rename('documents').reset_index())

,document_type,retrieval_stratum,documents
0,10-K,high,4
1,10-K,low,5
2,10-K,medium,4
3,10-Q,high,4
4,10-Q,low,4
5,10-Q,medium,4


## 4. Preflight volume, cost, and information-boundary audit

Both prompts receive identical passages. Enter current model prices above if a dollar estimate is required; otherwise the notebook reports token volume without inventing a price.

In [9]:
sample['retrieved_characters'] = sample.document_id.map(lambda x: sum(len(p['text']) for p in lookup[x]['passages']))
sample['approx_passage_tokens'] = np.ceil(sample.retrieved_characters/4).astype(int)
prompt_tokens = {v:int(np.ceil(len(x['prompt'])/4)) for v,x in loaded.items()}
approx_input = sum(int(sample.approx_passage_tokens.sum()) + CALIBRATION_N*prompt_tokens[v] for v in loaded)
assumed_output = CALIBRATION_N*2*900
cost = np.nan if pd.isna(INPUT_USD_PER_MILLION) or pd.isna(OUTPUT_USD_PER_MILLION) else approx_input/1e6*INPUT_USD_PER_MILLION + assumed_output/1e6*OUTPUT_USD_PER_MILLION
preflight=pd.DataFrame([{'documents':CALIBRATION_N,'calls':CALIBRATION_N*2,'approx_input_tokens':approx_input,'assumed_output_tokens':assumed_output,'estimated_usd':cost,'returns_loaded':False,'embeddings_loaded':False,'api_enabled':RUN_CALIBRATION}])
display(preflight); preflight.to_parquet(MANIFESTS/'calibration_preflight.parquet',index=False)

,documents,calls,approx_input_tokens,assumed_output_tokens,estimated_usd,returns_loaded,embeddings_loaded,api_enabled
0,25,50,189973,45000,NaN,False,False,True


## 5. Run the checkpointed paired API calibration

Set `RUN_CALIBRATION=True` only after reviewing Sections 1–4. The same payload is sent once under each frozen contract. `store=False` is explicit.

In [12]:
def payload_for(record):
    return {'firm_id':str(record['firm_id']),'document_id':str(record['document_id']),'document_type':record['document_type'],'available_date':str(record['available_date']),'theme':{'name':'critical_minerals_secure_supply','shock':'Sustained increase in strategic demand and policy support with Western supply-chain security and reorientation'},'passages':record['passages']}
if RUN_CALIBRATION:
    assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY without printing it.'
    from openai import OpenAI
    client=OpenAI()
    for version in ['v1','v2']:
        outdir=RESPONSES/version; outdir.mkdir(parents=True,exist_ok=True)
        for number,row in enumerate(sample.itertuples(index=False),1):
            path=outdir/f'{row.document_id}.json'
            if path.exists(): continue
            record=lookup[str(row.document_id)]; payload=payload_for(record)
            response=client.responses.create(model=OPENAI_MODEL,instructions=loaded[version]['prompt'],input=json.dumps(payload),store=False,reasoning={'effort':'low'},text={'format':{'type':'json_schema','name':f'critical_minerals_{version}','strict':True,'schema':loaded[version]['schema']}})
            envelope={'result':json.loads(response.output_text),'response_id':response.id,'model':response.model,'usage':response.usage.model_dump() if response.usage else None,'prompt_version':version,'prompt_sha256':contract_audit.set_index('version').loc[version,'prompt_sha256'],'source_content_sha256':record['content_sha256']}
            path.write_text(json.dumps(envelope,indent=2)); print(f'{version} {number}/{CALIBRATION_N}: {row.document_id}')
else:
    print('Calibration API calls disabled. Review the frozen sample and set RUN_CALIBRATION=True when ready.')

v1 1/25: 0001558370-24-013758
v1 2/25: 0001013462-24-000007
v1 3/25: 0001002910-24-000056
v1 4/25: 0000814547-24-000016
v1 5/25: 0001596532-24-000368
v1 6/25: 0001002910-24-000101
v1 7/25: 0000935703-24-000011
v1 8/25: 0001116132-24-000018
v1 9/25: 0001037646-24-000007
v1 10/25: 0001326160-24-000167
v1 11/25: 0000879169-24-000149
v1 12/25: 0000827054-24-000139
v1 13/25: 0000723254-24-000036
v1 14/25: 0000093556-24-000032
v1 15/25: 0001524472-24-000006
v1 16/25: 0000927066-24-000128
v1 17/25: 0001601046-24-000061
v1 18/25: 0001164727-24-000039
v1 19/25: 0000804328-23-000055
v1 20/25: 0001108524-24-000005
v1 21/25: 0001274494-24-000004
v1 22/25: 0001564708-24-000070
v1 23/25: 0000950170-24-120749
v1 24/25: 0000915913-24-000158
v1 25/25: 0000950170-23-027948
v2 1/25: 0001558370-24-013758
v2 2/25: 0001013462-24-000007
v2 3/25: 0001002910-24-000056
v2 4/25: 0000814547-24-000016
v2 5/25: 0001596532-24-000368
v2 6/25: 0001002910-24-000101
v2 7/25: 0000935703-24-000011
v2 8/25: 0001116132-24-0

## 6. Validate evidence and compare behavior

A response is usable only when each quotation occurs exactly in a supplied passage, contains at most 60 words, and—under v2—references the passage that contains it. Higher label volume is not treated as better performance.

In [13]:
audit_rows=[]; exposure_rows=[]
for version in ['v1','v2']:
    for row in sample.itertuples(index=False):
        path=RESPONSES/version/f'{row.document_id}.json'
        if not path.exists(): continue
        result=json.loads(path.read_text())['result']; rec=lookup[str(row.document_id)]; passage_map={p['passage_id']:p['text'] for p in rec['passages']}
        quote_valid=[]
        for exposure in result.get('exposures',[]):
            quote=exposure.get('evidence_quote',''); pid=exposure.get('evidence_passage_id','')
            exact=(quote in passage_map.get(pid,'')) if version=='v2' else any(quote in text for text in passage_map.values())
            valid=bool(quote and exact and len(quote.split())<=60); quote_valid.append(valid)
            exposure_rows.append({'version':version,'document_id':str(row.document_id),'quote_valid':valid,**exposure})
        consistent=(bool(result.get('has_exposure_evidence')) == (len(result.get('exposures',[]))>0))
        audit_rows.append({'version':version,'document_id':str(row.document_id),'document_type':row.document_type,'has_exposure':bool(result.get('has_exposure_evidence')),'exposures':len(result.get('exposures',[])),'all_quotes_valid':all(quote_valid) if quote_valid else True,'internally_consistent':consistent})
audit=pd.DataFrame(audit_rows); exposures=pd.DataFrame(exposure_rows)
if len(audit):
    summary=audit.groupby('version',as_index=False).agg(documents=('document_id','nunique'),exposure_rate=('has_exposure','mean'),mean_exposures=('exposures','mean'),exact_quote_rate=('all_quotes_valid','mean'),consistency_rate=('internally_consistent','mean'))
    display(summary); display(pd.crosstab(audit.query("version=='v1'").set_index('document_id').has_exposure,audit.query("version=='v2'").set_index('document_id').has_exposure,rownames=['v1'],colnames=['v2']))
    audit.to_parquet(CAL/'automated_response_audit.parquet',index=False); exposures.to_parquet(CAL/'exposure_detail.parquet',index=False)
else: print('No paired responses yet.')

,version,documents,exposure_rate,mean_exposures,exact_quote_rate,consistency_rate
0,v1,25,0.44,0.72,1.00,1.0
1,v2,25,0.44,0.76,0.96,1.0


v2,False,True
v1,,
False,14,0
True,0,11


## 7. Create a blinded human-adjudication packet

Automated checks cannot establish economic correctness. Prompt labels are randomized to A/B per document and the key is stored separately. Reviewers judge unsupported inference, missed material evidence, causal coherence, direction, materiality, and quotation quality before unblinding.

In [14]:
if len(audit)==CALIBRATION_N*2:
    rng=np.random.default_rng(CALIBRATION_SEED); packet=[]; key=[]
    for doc in sample.document_id.astype(str):
        order=['v1','v2'] if rng.integers(0,2)==0 else ['v2','v1']
        for label,version in zip(['A','B'],order):
            result=json.loads((RESPONSES/version/f'{doc}.json').read_text())['result']
            packet.append({'document_id':doc,'candidate':label,'has_exposure':result['has_exposure_evidence'],'assessment':result.get('document_assessment',result.get('document_summary','')),'exposures_json':json.dumps(result['exposures'],sort_keys=True),'unsupported_inference':'','missed_material_evidence':'','direction_correct':'','materiality_correct':'','causal_chain_coherent':'','reviewer_notes':''})
            key.append({'document_id':doc,'candidate':label,'prompt_version':version})
    pd.DataFrame(packet).to_csv(CAL/'blinded_human_review.csv',index=False)
    pd.DataFrame(key).to_parquet(MANIFESTS/'restricted_blinding_key.parquet',index=False)
    print('Created blinded_human_review.csv. Complete it before inspecting the restricted key.')
else: print('The packet is created only after all 50 paired responses exist.')

Created blinded_human_review.csv. Complete it before inspecting the restricted key.


## 8. Decision rule and handoff

Do **not** choose v2 because it finds more firms. Advance it only if schema/ontology consistency is 100%, exact-quote validity is at least 95%, and blinded review shows no increase in unsupported causal claims while improving or preserving material-evidence recall. Freeze the winning prompt, schema, model identifier, hashes, eligibility rules, and aggregation contract before the full S&P 500 extraction. Portfolio construction and forward returns remain inaccessible until that freeze.